# PAD-UFES-20 Class-Aware Augmentation Training

Run this notebook in Google Colab with a GPU runtime. It clones the repo, prepares PAD-UFES-20, checks DagsHub MLflow, runs the ISIC-initialized multimodal class-aware augmentation experiment, and compares the candidate against the current best metrics.

## Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Setup

In [ ]:
from pathlib import Path
import json
import os
import subprocess

try:
    from google.colab import userdata
except ImportError:
    userdata = None


def get_config(name, default=None):
    value = os.environ.get(name)
    if value:
        return value
    if userdata is not None:
        try:
            return userdata.get(name) or default
        except Exception:
            return default
    return default


def export_config(name, default=None, required=False):
    value = get_config(name, default)
    if required and not value:
        raise RuntimeError(f'Set {name} in Colab Secrets before training.')
    if value:
        os.environ[name] = str(value)
    return value


REPO_URL = 'https://github.com/SalmaneSossey/mlops-teledermatology.git'
BRANCH = 'main'
REPO_DIR = Path('/content/mlops-teledermatology')
HF_DATASET_REPO = get_config('PAD_UFES20_HF_REPO_ID', 'SalmaneExploring/pad-ufes-20')
DATA_ROOT = Path('/content/pad_ufes_20')
IMAGES_DIR = DATA_ROOT / 'all_images'
SPLITS_DIR = Path('data/processed/splits')
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/mlops-teledermatology')
ISIC_CHECKPOINT = DRIVE_PROJECT_DIR / 'runs/isic_2019_pretrain/efficientnet_b0_best.pt'
OUTPUT_DIR = DRIVE_PROJECT_DIR / 'runs/multimodal_class_aware_aug/isic_init'
CANDIDATE_BUNDLE_DIR = DRIVE_PROJECT_DIR / 'model_bundles/class_aware_candidate'

RUN_TRAINING = True
BUILD_CANDIDATE_BUNDLE = False
EPOCHS = 8
BATCH_SIZE = 32
ALLOW_CPU = False
EXPERIMENT_NAME = 'pad-ufes-20-multimodal-isic-class-aware-aug'

CURRENT_BEST = {
    'test_macro_f1': 0.6902,
    'test_balanced_accuracy': 0.6804,
    'test_high_risk_recall': 0.8902,
    'SCC_recall': 0.2069,
}
TOLERANCE = 0.02

export_config('DAGSHUB_TOKEN', required=True)
export_config('DAGSHUB_USERNAME')
export_config('DAGSHUB_REPO_OWNER', 'SalmaneSossey')
export_config('DAGSHUB_REPO_NAME', 'mlops-teledermatology')
export_config('DAGSHUB_MLFLOW_TRACKING_URI')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CANDIDATE_BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    subprocess.run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], cwd='/content', check=True)

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())
print('Hugging Face dataset:', HF_DATASET_REPO)
print('Output dir:', OUTPUT_DIR)
print('ISIC checkpoint:', ISIC_CHECKPOINT)
print('MLflow tracking URI:', os.environ.get('DAGSHUB_MLFLOW_TRACKING_URI') or f"https://dagshub.com/{os.environ['DAGSHUB_REPO_OWNER']}/{os.environ['DAGSHUB_REPO_NAME']}.mlflow")


## Install Dependencies

In [ ]:
!pip -q install mlflow huggingface_hub scikit-learn


## DagsHub And GPU Checks

In [ ]:
import mlflow
import torch
from mlflow.tracking import MlflowClient

tracking_uri = os.environ.get('DAGSHUB_MLFLOW_TRACKING_URI') or f"https://dagshub.com/{os.environ['DAGSHUB_REPO_OWNER']}/{os.environ['DAGSHUB_REPO_NAME']}.mlflow"
mlflow.set_tracking_uri(tracking_uri)
experiments = MlflowClient(tracking_uri).search_experiments()
print('DagsHub token available:', bool(os.environ.get('DAGSHUB_TOKEN')))
print('MLflow tracking URI:', tracking_uri)
print('Existing experiments:', [(experiment.experiment_id, experiment.name) for experiment in experiments])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
elif RUN_TRAINING and not ALLOW_CPU:
    raise RuntimeError('Select a Colab GPU runtime before starting training.')

if not ISIC_CHECKPOINT.exists():
    raise FileNotFoundError(f'Missing ISIC checkpoint: {ISIC_CHECKPOINT}')


## Download Data And Build Splits

In [ ]:
!python -m src.data.download_pad_ufes_20 \
  --repo-id "{HF_DATASET_REPO}" \
  --output-dir "{DATA_ROOT}" \
  --force

!python -m src.data.make_image_splits \
  --metadata-path "{DATA_ROOT / 'metadata.csv'}" \
  --images-dir "{IMAGES_DIR}" \
  --output-dir "{SPLITS_DIR}"


## Train Class-Aware Multimodal Candidate

In [ ]:
if RUN_TRAINING:
    command = [
        'python', '-m', 'src.training.train_multimodal_baseline',
        '--images-dir', str(IMAGES_DIR),
        '--metadata-path', str(DATA_ROOT / 'metadata.csv'),
        '--splits-dir', str(SPLITS_DIR),
        '--output-dir', str(OUTPUT_DIR),
        '--experiment-name', EXPERIMENT_NAME,
        '--hf-dataset-repo', HF_DATASET_REPO,
        '--initial-image-checkpoint', str(ISIC_CHECKPOINT),
        '--sampler', 'weighted_random',
        '--augment-strength', 'class_aware',
        '--epochs', str(EPOCHS),
        '--batch-size', str(BATCH_SIZE),
    ]
    if ALLOW_CPU:
        command.append('--allow-cpu')
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
else:
    subprocess.run(['python', '-m', 'src.training.train_multimodal_baseline', '--help'], check=True)


If Colab runs out of memory, set `BATCH_SIZE = 16` in the setup cell and rerun from the runtime check onward.

## Compare Candidate Metrics

In [ ]:
import pandas as pd

metrics_path = OUTPUT_DIR / 'multimodal_test_metrics.json'
report_path = OUTPUT_DIR / 'multimodal_classification_report.csv'
if not metrics_path.exists():
    raise FileNotFoundError(f'Missing metrics file: {metrics_path}')
if not report_path.exists():
    raise FileNotFoundError(f'Missing classification report: {report_path}')

metrics = json.loads(metrics_path.read_text())
report = pd.read_csv(report_path, index_col=0)
scc_recall = float(report.loc['SCC', 'recall'])

comparison = pd.DataFrame(
    [
        {
            'metric': 'macro F1',
            'current_best': CURRENT_BEST['test_macro_f1'],
            'candidate': float(metrics['test_macro_f1']),
            'pass_rule': float(metrics['test_macro_f1']) >= CURRENT_BEST['test_macro_f1'] - TOLERANCE,
        },
        {
            'metric': 'balanced accuracy',
            'current_best': CURRENT_BEST['test_balanced_accuracy'],
            'candidate': float(metrics['test_balanced_accuracy']),
            'pass_rule': float(metrics['test_balanced_accuracy']) >= CURRENT_BEST['test_balanced_accuracy'] - TOLERANCE,
        },
        {
            'metric': 'high-risk recall',
            'current_best': CURRENT_BEST['test_high_risk_recall'],
            'candidate': float(metrics['test_high_risk_recall']),
            'pass_rule': float(metrics['test_high_risk_recall']) >= CURRENT_BEST['test_high_risk_recall'] - TOLERANCE,
        },
        {
            'metric': 'SCC recall',
            'current_best': CURRENT_BEST['SCC_recall'],
            'candidate': scc_recall,
            'pass_rule': scc_recall > CURRENT_BEST['SCC_recall'],
        },
    ]
)
display(comparison)
display(report.loc[['BCC', 'MEL', 'SCC'], ['precision', 'recall', 'f1-score', 'support']])

if bool(comparison['pass_rule'].all()):
    print('Candidate passes the class-aware augmentation decision rule.')
else:
    print('Candidate should be reported as an ablation, not promoted as the final model.')


## Optional Candidate Bundle

In [ ]:
if BUILD_CANDIDATE_BUNDLE:
    command = [
        'python', '-m', 'src.inference.build_multimodal_bundle',
        '--metadata-path', str(DATA_ROOT / 'metadata.csv'),
        '--splits-dir', str(SPLITS_DIR),
        '--output-dir', str(CANDIDATE_BUNDLE_DIR),
        '--checkpoint-path', str(OUTPUT_DIR / 'efficientnet_b0_multimodal_best.pt'),
        '--mlflow-run-id', 'class-aware-colab-candidate',
    ]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
else:
    print('Skipping bundle build. Set BUILD_CANDIDATE_BUNDLE = True after the candidate passes review.')
